# COS801 RQ7.1 — Model compression benchmark
This notebook runs the same leakage-audited split through FP32 baselines, INT8 quantisation, structured pruning, and knowledge distillation. Start with smoke mode; only full-mode results belong in the report.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed

In [ ]:
# Runtime code lives in /content; Drive is used only for data and results.
REPO_URL = 'https://github.com/vbhaga89/COS801_Project.git'
BRANCH = 'feature/rq7.1-model-compression'
DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO_ROOT = Path('/content/COS801_Project_runtime')
PROJECT_ROOT = REPO_ROOT / 'rq7.1-model-compression'
REAL_DATA_DIR = DRIVE_ROOT / 'COS801_data' / 'plantvillage'
SMOKE_DATA_DIR = DRIVE_ROOT / 'COS801_data' / 'rq71_smoke'
OUTPUT_ROOT = DRIVE_ROOT / 'COS801_results' / 'rq7.1'
SMOKE_MODE = True

# Clean only the temporary runtime clone, never your Drive files.
if not PROJECT_ROOT.exists():
    if REPO_ROOT.exists():
        print(f'Removing incomplete runtime clone: {REPO_ROOT}')
        shutil.rmtree(REPO_ROOT)
    print(f'Cloning project into runtime: {REPO_ROOT}')
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)])

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}')

DATA_DIR = SMOKE_DATA_DIR if SMOKE_MODE else REAL_DATA_DIR
OUTPUT_DIR = OUTPUT_ROOT / ('smoke' if SMOKE_MODE else 'full')
CONFIG = PROJECT_ROOT / 'configs' / ('smoke.json' if SMOKE_MODE else 'full.json')

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_DIR     =', DATA_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)
print('CONFIG       =', CONFIG)

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'torch-pruning', 'scikit-learn'])
os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

In [ ]:
# Keep the shared configuration reproducible while supplying machine-specific paths.
if SMOKE_MODE:
    run([sys.executable, 'make_smoke_dataset.py', '--output', str(DATA_DIR)])
elif not DATA_DIR.exists():
    raise FileNotFoundError(f'Real dataset folder not found: {DATA_DIR}')

cfg = json.loads(CONFIG.read_text())
cfg['data_dir'] = str(DATA_DIR)
cfg['output_dir'] = str(OUTPUT_DIR)
RUNTIME_CONFIG = Path('/content/rq71_runtime_config.json')
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2))
print('Runtime config written to:', RUNTIME_CONFIG)
cfg

## 1. EDA and leakage gate

In [ ]:
run([sys.executable, 'run_experiment.py', 'eda', '--config', str(RUNTIME_CONFIG)])

## 2. FP32 baselines

In [ ]:
run([sys.executable, 'run_experiment.py', 'train', '--config', str(RUNTIME_CONFIG), '--model', 'mobilenet_v3_small'])
run([sys.executable, 'run_experiment.py', 'train', '--config', str(RUNTIME_CONFIG), '--model', 'efficientnet_b0'])

## 3. Compression experiments
Select the better validation baseline as the teacher/checkpoint before a full run.

In [ ]:
TEACHER = Path(cfg['output_dir']) / 'mobilenet_v3_small' / 'baseline.pt'
run([sys.executable, 'run_experiment.py', 'quantize', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])
run([sys.executable, 'run_experiment.py', 'prune', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])
run([sys.executable, 'run_experiment.py', 'distill', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(TEACHER)])

## 4. Results table

In [ ]:
import pandas as pd
results = pd.read_csv(Path(cfg['output_dir']) / 'metrics.csv')
results.sort_values(['accuracy', 'model_size_mb'], ascending=[False, True])